# Diabetes Risk Prediction — Notebook Técnico

> Predição de risco de Diabetes Mellitus Tipo 2 utilizando algoritmos supervisionados de Machine Learning.

Este notebook documenta o pipeline técnico completo: configuração do ambiente, carregamento dos dados, treinamento dos modelos, avaliação e interpretação dos resultados.

---

## Sumário

1. [Configuração do Ambiente](#1-configuração-do-ambiente)
2. [Carregamento dos Dados](#2-carregamento-dos-dados)
3. [Inspeção dos Dados Processados](#3-inspeção-dos-dados-processados)
4. [Análise Exploratória de Dados](#4-análise-exploratória-de-dados)
5. [Treinamento dos Modelos](#5-treinamento-dos-modelos)
6. [Avaliação dos Modelos](#6-avaliação-dos-modelos)
7. [Comparação de Desempenho](#7-comparação-de-desempenho)
8. [Importância das Variáveis](#8-importância-das-variáveis)
9. [Discussão Clínica](#9-discussão-clínica)
10. [Conclusão](#10-conclusão)

---

## 1. Configuração do Ambiente

Importação das bibliotecas necessárias e configuração dos módulos internos do projeto.

O diretório raiz é adicionado ao `sys.path` para permitir importação dos módulos em `src/` independentemente do diretório de execução.

In [ ]:
import os
import sys
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

# Raiz do projeto
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)

# Módulos internos
from src.train import get_model_configs, RANDOM_STATE
from src.evaluate import (
    compute_metrics,
    plot_confusion_matrices,
    plot_roc_curves,
    plot_feature_importance,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Random State : {RANDOM_STATE}")

---

## 2. Carregamento dos Dados

Os dados já passaram pelas etapas de:

- análise exploratória
- avaliação de qualidade
- tratamento de valores ausentes
- engenharia de atributos
- padronização (`StandardScaler`)
- divisão treino/teste (80/20, estratificada)

Os arquivos processados são carregados diretamente de `data/processed/`.

**Controle de Data Leakage:**  
A divisão foi realizada antes de qualquer transformação. Imputação e padronização foram ajustadas exclusivamente no conjunto de treino e aplicadas ao teste sem re-ajuste.

In [ ]:
PROCESSED_DIR = os.path.join("data", "processed")

X_train = pd.read_csv(os.path.join(PROCESSED_DIR, "X_train.csv"), index_col=0)
X_test  = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test.csv"),  index_col=0)

y_train = (
    pd.read_csv(os.path.join(PROCESSED_DIR, "y_train.csv"), index_col=0)
    .squeeze()
)
y_test = (
    pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv"), index_col=0)
    .squeeze()
)

print(f"Treino : {X_train.shape}")
print(f"Teste  : {X_test.shape}")
print()
print(f"Pacientes positivos — Treino : {y_train.mean():.1%}")
print(f"Pacientes positivos — Teste  : {y_test.mean():.1%}")

---

## 3. Inspeção dos Dados Processados

Verificação rápida da integridade dos dados após o carregamento.

In [ ]:
print("=== Primeiras linhas — X_train ===")
display(X_train.head())

In [ ]:
print("=== Estatísticas descritivas — X_train ===")
display(X_train.describe().round(4))

In [ ]:
print("=== Variáveis disponíveis para treinamento ===")
print()

for i, feature in enumerate(X_train.columns, start=1):
    print(f"{i:2d}. {feature}")

print(f"\nTotal de atributos : {X_train.shape[1]}")
print("Variável alvo      : Outcome")

In [ ]:
print("=== Tipos de dados ===")
display(X_train.dtypes.to_frame(name="dtype"))

In [ ]:
print("=== Valores ausentes após pré-processamento ===")
missing = X_train.isnull().sum()
display(missing[missing > 0].to_frame(name="ausentes") if missing.sum() > 0 else "Nenhum valor ausente encontrado.")

---

## 4. Análise Exploratória de Dados

As visualizações a seguir foram geradas durante a etapa de EDA e são exibidas aqui para contextualizar a modelagem.

In [8]:
FIGURES_DIR = os.path.join("outputs", "figures")

def show_figure(filename, title="", figsize=(12, 6)):
    """Exibe uma figura salva em outputs/figures/."""
    path = os.path.join(FIGURES_DIR, filename)
    img = plt.imread(path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=13, pad=10)
    plt.tight_layout()
    plt.show()

In [ ]:
show_figure(
    "01_class_distribution.png",
    "Figura 1 — Distribuição das Classes (Outcome)",
    figsize=(8, 5),
)

**Observação:** O dataset apresenta desbalanceamento entre as classes — aproximadamente 65% não diabéticos e 35% diabéticos. Esse desequilíbrio é considerado na interpretação das métricas, especialmente Precision e Recall.

In [ ]:
show_figure(
    "02_feature_distributions.png",
    "Figura 2 — Distribuição das Variáveis Preditoras",
    figsize=(14, 8),
)

**Observação:** `Insulin` e `DiabetesPedigreeFunction` apresentam assimetria positiva acentuada. `Glucose` e `BMI` exibem distribuições aproximadamente normais.

In [ ]:
show_figure(
    "03_boxplots_by_class.png",
    "Figura 3 — Boxplots das Variáveis por Classe",
    figsize=(14, 8),
)

**Observação:** `Glucose` e `BMI` apresentam separação clara entre as classes — candidatas naturais a preditores relevantes. `BloodPressure` exibe menor poder discriminativo.

In [ ]:
show_figure(
    "04_correlation_heatmap.png",
    "Figura 4 — Mapa de Correlação (Pearson)",
    figsize=(9, 7),
)

**Observação:** `Glucose` apresenta a maior correlação com `Outcome` (~0.47). `Age` e `BMI` seguem como variáveis correlacionadas ao desfecho. Não há multicolinearidade severa entre os preditores.

In [ ]:
show_figure(
    "05_missing_data_pattern.png",
    "Figura 5 — Padrão de Dados Ausentes",
    figsize=(9, 6),
)

**Observação:** `Insulin` e `SkinThickness` concentram a maior taxa de ausências (zeros biologicamente impossíveis). A imputação foi realizada com mediana estratificada pela classe, ajustada exclusivamente no conjunto de treino.

In [ ]:
show_figure(
    "06_pairplot_key_features.png",
    "Figura 6 — Pairplot das Variáveis-Chave",
    figsize=(12, 10),
)

**Observação:** A combinação `Glucose × BMI` exibe a melhor separabilidade visual entre as classes. Nenhuma combinação bivariada separa perfeitamente os grupos — justificando o uso de modelos multivariados.

---

## 5. Treinamento dos Modelos

Três algoritmos foram selecionados para comparação, representando estratégias distintas de aprendizado:

| Modelo | Estratégia | Interpretabilidade |
|--------|------------|-------------------|
| Logistic Regression | Linear | Alta |
| Decision Tree | Não linear / Árvore | Alta |
| Random Forest | Ensemble de árvores | Média |

Todos os hiperparâmetros são definidos em `src/train.py` para garantir consistência entre execuções.

In [ ]:
configs = get_model_configs()

print("=== Configurações dos modelos ===")
print()
for name, cfg in configs.items():
    print(f"{name}")
    print(f"  {cfg['model']}")
    print()

### 5.1 Logistic Regression

Modelo linear de referência. Coeficientes interpretáveis como contribuição ao log-odds do desfecho.

In [ ]:
log_reg_model = configs["LogisticRegression"]["model"]
log_reg_model.fit(X_train, y_train)

print(log_reg_model)
print()

# Coeficientes
coef_df = (
    pd.DataFrame(
        {"Feature": X_train.columns, "Coeficiente": log_reg_model.coef_[0]}
    )
    .sort_values("Coeficiente", ascending=False)
    .reset_index(drop=True)
)

print("=== Coeficientes — Logistic Regression ===")
display(coef_df)

### 5.2 Decision Tree

In [ ]:
dt_model = configs["DecisionTree"]["model"]
dt_model.fit(X_train, y_train)

print(dt_model)
print()
print(f"Profundidade máxima utilizada : {dt_model.get_depth()}")
print(f"Número de folhas              : {dt_model.get_n_leaves()}")

### 5.3 Random Forest

In [ ]:
rf_model = configs["RandomForest"]["model"]
rf_model.fit(X_train, y_train)

print(rf_model)
print()
print(f"Número de estimadores : {rf_model.n_estimators}")
print(f"Features por split    : {rf_model.max_features}")

In [ ]:
models = {
    "LogisticRegression": log_reg_model,
    "DecisionTree"      : dt_model,
    "RandomForest"      : rf_model,
}

print("Modelos treinados:", list(models.keys()))

---

## 6. Avaliação dos Modelos

A avaliação é realizada **exclusivamente sobre o conjunto de teste** — dados nunca vistos durante o treinamento.

### Métricas utilizadas

| Métrica | Fórmula | Relevância Clínica |
|---------|---------|--------------------|
| **Accuracy** | (VP + VN) / Total | Visão geral do desempenho |
| **Precision** | VP / (VP + FP) | Qualidade dos positivos preditos |
| **Recall** | VP / (VP + FN) | **Prioritária** — detecta diabéticos reais |
| **F1-score** | 2 × (P × R) / (P + R) | Equilíbrio entre Precision e Recall |
| **ROC-AUC** | Área sob a curva ROC | Capacidade discriminativa geral |

> **Atenção clínica:** Falso Negativo (FN) = diabético classificado como saudável. Implica atraso no diagnóstico e progressão silenciosa da doença — impacto mais grave que o Falso Positivo.

In [ ]:
metrics_df = compute_metrics(models, X_test, y_test)

print("=== Métricas de Desempenho — Conjunto de Teste ===")
display(metrics_df.round(4))

### 6.1 Classification Report por Modelo

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"{'='*55}")
    print(f" {name}")
    print(f"{'='*55}")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Não diabético", "Diabético"],
        )
    )

### 6.2 Matrizes de Confusão

In [ ]:
plot_confusion_matrices(models, X_test, y_test)

In [ ]:
show_figure(
    "07_confusion_matrices.png",
    "Figura 7 — Matrizes de Confusão",
    figsize=(14, 5),
)

### 6.3 Curvas ROC

In [ ]:
plot_roc_curves(models, X_test, y_test)

In [ ]:
show_figure(
    "08_roc_curves.png",
    "Figura 8 — Curvas ROC",
    figsize=(9, 7),
)

---

## 7. Comparação de Desempenho

In [ ]:
show_figure(
    "10_metrics_comparison.png",
    "Figura 10 — Comparação de Desempenho entre Modelos",
    figsize=(12, 6),
)

In [ ]:
print("=== Ranking por Recall (Sensibilidade) ===")
display(
    metrics_df.sort_values("Recall", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

In [ ]:
print("=== Ranking por ROC-AUC ===")
display(
    metrics_df.sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

---

## 8. Importância das Variáveis

Modelos baseados em árvores fornecem estimativas da contribuição relativa de cada variável ao processo de decisão.

Essa análise permite verificar se os preditores mais relevantes são compatíveis com a literatura médica — validando a hipótese **H1**.

In [ ]:
plot_feature_importance(models, X_test.columns.tolist())

In [ ]:
show_figure(
    "09_feature_importance.png",
    "Figura 9 — Importância das Variáveis (Random Forest)",
    figsize=(10, 6),
)

In [ ]:
importances_df = (
    pd.DataFrame(
        {
            "Feature"   : X_train.columns,
            "Importance": rf_model.feature_importances_,
        }
    )
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

importances_df["Importance (%)"] = (importances_df["Importance"] * 100).round(2)
importances_df["Importância acumulada (%)"] = importances_df["Importance (%)"].cumsum().round(2)

print("=== Ranking de Importância das Variáveis — Random Forest ===")
display(importances_df)

### Interpretação

| Variável | Relevância clínica esperada |
|----------|-----------------------------|
| **Glucose** | Principal marcador bioquímico do diagnóstico de DM2 |
| **BMI** | Associado à obesidade e resistência à insulina |
| **Age** | Risco crescente de DM2 com a idade |
| **DiabetesPedigreeFunction** | Predisposição genética e histórico familiar |
| **Insulin** | Marcador direto da função pancreática |

> Importância estatística ≠ causalidade biológica. A interpretação deve sempre ser ancorada em evidências clínicas e epidemiológicas.

---

## 9. Discussão Clínica

### Verificação das Hipóteses

| Hipótese | Resultado |
|----------|----------|
| **H1** — Glucose, BMI, Age e DiabetesPedigreeFunction entre os principais preditores | Verificar ranking acima |
| **H2** — Random Forest com desempenho ≥ Logistic Regression | Verificar tabela de métricas |

### Aplicabilidade Clínica

**Uso potencial em triagem:**  
O modelo pode identificar indivíduos de maior risco e direcioná-los para investigação complementar — contribuindo para priorização de exames, encaminhamento especializado e monitoramento proativo.

**Limitações de aplicação:**  
O modelo não substitui avaliação médica. O diagnóstico definitivo exige histórico clínico completo, exame físico e exames laboratoriais adicionais.

**Contexto de uso recomendado:**  
Clinical Decision Support Systems (CDSS) integrados a prontuários eletrônicos, com o modelo operando como sinal de alerta — nunca como instrumento diagnóstico autônomo.

### Limitações

- Dataset restrito a mulheres de um único grupo étnico (n = 768)
- Valores ausentes imputados — incerteza adicional
- Sem validação cruzada, externa ou multicêntrica
- Resultados representam evidência preliminar

---

## 10. Conclusão

Este notebook documenta o pipeline técnico completo para predição de Diabetes Mellitus Tipo 2.

**Boas práticas aplicadas:**
- Controle rigoroso de data leakage (divisão antes das transformações)
- Avaliação exclusivamente sobre dados não vistos
- Métricas clínicas como critério de comparação (Recall prioritário)
- Interpretação ancorada na literatura médica
- Código modular e reprodutível

**Próximos passos:**
- Validação cruzada estratificada (k = 5 ou k = 10)
- Otimização de hiperparâmetros (GridSearchCV / RandomizedSearchCV)
- Calibração probabilística (Platt Scaling / Isotonic Regression)
- Explicabilidade com SHAP e LIME
- Avaliação de algoritmos de gradient boosting (XGBoost, LightGBM, CatBoost)

---

**Referências:**
- International Diabetes Federation (IDF). *IDF Diabetes Atlas*.
- National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK).
- Pima Indians Diabetes Database — UCI Machine Learning Repository.
- Scikit-learn Documentation — https://scikit-learn.org
- Pandas Documentation — https://pandas.pydata.org
- NumPy Documentation — https://numpy.org
- Matplotlib Documentation — https://matplotlib.org